In [ ]:
# Install StyleGAN2-ADA PyTorch
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch

# Requirements
!pip install torch torchvision
!pip install ninja imageio imageio-ffmpeg
!pip install click requests tqdm pyspng

In [ ]:
# Download pretrained 256×256 model
!wget https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl

# This model is trained on 70,000 faces
# We'll fine-tune it on our 1,250 hijabi faces

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
from PIL import Image

def prepare_stylegan_dataset_all():
    SOURCE = '/content/drive/MyDrive/data'
    OUTPUT = '/content/stylegan_dataset'
    emotions = ['happy', 'sad', 'angry', 'surprise', 'neutral']
    valid_ext = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

    for emotion in emotions:
        src_folder = os.path.join(SOURCE, emotion)
        dst_folder = os.path.join(OUTPUT, emotion)
        os.makedirs(dst_folder, exist_ok=True)

        i = 0
        for name in sorted(os.listdir(src_folder)):
            path = os.path.join(src_folder, name)

            if os.path.isdir(path) or name.startswith('.'):
                continue
            if not name.lower().endswith(valid_ext):
                print(f"⏭️ [{emotion}] Skipping non-image: {name}")
                continue

            try:
                img = Image.open(path).convert('RGB')
                img = img.resize((256, 256), Image.LANCZOS)
                dst = os.path.join(dst_folder, f"{emotion}_{i:04d}.png")
                img.save(dst, "PNG")
                i += 1
            except Exception as e:
                print(f"❌ [{emotion}] Failed: {name} -> {e}")

        print(f"✅ [{emotion}] Converted {i} images")

prepare_stylegan_dataset_all()


In [ ]:
!pip install torch torchvision


In [ ]:
!pip uninstall -y torch torchvision torchaudio torchtext functorch


In [ ]:
!pip install --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio


In [ ]:

!pip install timm  # For pretrained models
!pip install lpips  # For quality metrics
!pip install ninja  # For StyleGAN2 compilation

In [ ]:
# Cell 2: Download StyleGAN2-ADA
import os

if not os.path.exists('stylegan2-ada-pytorch'):
    !git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git

os.chdir('stylegan2-ada-pytorch')

# Verify
!ls

In [ ]:
# Cell 3: Download FFHQ pretrained
import os
import gdown

# FFHQ 256×256 pretrained model
if not os.path.exists('ffhq256.pkl'):
    # Option 1: Direct download (if available)
    url = 'https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl'
    !wget {url} -O ffhq256.pkl

    # Option 2: Google Drive (backup)
    # file_id = 'YOUR_FILE_ID'
    # gdown.download(f'https://drive.google.com/uc?id={file_id}', 'ffhq256.pkl', quiet=False)

print("✅ Pretrained model downloaded!")

In [ ]:
# Cell 4: Prepare dataset in StyleGAN2 format

import os
import shutil
from PIL import Image
from tqdm import tqdm

def prepare_stylegan_dataset(
    source_path='/content/drive/MyDrive/data',
    output_path='/content/stylegan_data',
    img_size=256
):
    """
    Convert dataset to StyleGAN2 format:

    stylegan_data/
        00000/
            img00000000.png
            img00000001.png
            ...

    (All images in one folder, numbered sequentially)
    """

    emotions = ['happy', 'sad', 'angry', 'surprise', 'neutral']

    # Create output directory
    os.makedirs(output_path, exist_ok=True)
    img_folder = os.path.join(output_path, '00000')
    os.makedirs(img_folder, exist_ok=True)

    # Also save emotion labels separately
    labels_file = os.path.join(output_path, 'dataset.json')

    labels_dict = {}
    img_counter = 0

    print("📂 Converting dataset...")

    for emotion_idx, emotion in enumerate(emotions):
        emotion_folder = os.path.join(source_path, emotion)

        if not os.path.exists(emotion_folder):
            print(f"⚠️ {emotion} folder not found, skipping")
            continue

        images = [f for f in os.listdir(emotion_folder)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        print(f"\n{emotion}: {len(images)} images")

        for img_name in tqdm(images, desc=emotion):
            # Read image
            img_path = os.path.join(emotion_folder, img_name)
            img = Image.open(img_path).convert('RGB')

            # Resize to target size
            img = img.resize((img_size, img_size), Image.LANCZOS)

            # Save with sequential numbering
            new_name = f'img{img_counter:08d}.png'
            new_path = os.path.join(img_folder, new_name)
            img.save(new_path, 'PNG')

            # Save label
            labels_dict[new_name] = emotion_idx

            img_counter += 1

    # Save labels as JSON
    import json
    with open(labels_file, 'w') as f:
        json.dump({
            'labels': [[name, label] for name, label in labels_dict.items()]
        }, f, indent=2)

    print(f"\n✅ Dataset prepared!")
    print(f"   Total images: {img_counter}")
    print(f"   Location: {output_path}")
    print(f"   Labels: {labels_file}")

    return output_path


# Run preparation
dataset_path = prepare_stylegan_dataset()

In [ ]:
# Cell 5: Load pretrained StyleGAN2

import pickle
import torch
import copy

def load_stylegan2_generator(pkl_path='ffhq256.pkl'):
    """
    Load pretrained StyleGAN2 generator
    """
    print(f"📦 Loading pretrained model from {pkl_path}...")

    with open(pkl_path, 'rb') as f:
        G = pickle.load(f)['G_ema'].cuda()

    print(f"✅ Generator loaded!")
    print(f"   Resolution: {G.img_resolution}×{G.img_resolution}")
    print(f"   Channels: {G.img_channels}")

    return G


def make_conditional(G_original, num_classes=5):
    """
    Convert unconditional generator to conditional

    This is a simplified version - adds emotion embedding
    """

    G = copy.deepcopy(G_original)

    # Add emotion embedding layer
    G.c_dim = num_classes  # Number of emotion classes

    # Modify mapping network to accept conditioning
    # (This is simplified - full implementation is complex)

    print(f"✅ Converted to conditional generator")
    print(f"   Classes: {num_classes}")

    return G


# Load model
try:
    generator = load_stylegan2_generator('ffhq256.pkl')
    print("\n✅ Ready to train!")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\n💡 Solution: Make sure ffhq256.pkl exists")

In [ ]:
# Cell 6: Simplified Conditional DCGAN for 256×256

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import json
from tqdm import tqdm

# ==========================================
# Dataset
# ==========================================

class ConditionalEmotionDataset(Dataset):
    """
    Dataset for conditional GAN
    """

    def __init__(self, data_path, img_size=256):
        self.data_path = data_path
        self.img_size = img_size

        # Load labels
        labels_file = os.path.join(data_path, 'dataset.json')
        with open(labels_file, 'r') as f:
            data = json.load(f)

        self.labels = data['labels']  # [[filename, emotion_idx], ...]

        self.transform = transforms.Compose([
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # [-1, 1]
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_name, emotion_idx = self.labels[idx]

        # Load image
        img_path = os.path.join(self.data_path, '00000', img_name)
        img = Image.open(img_path).convert('RGB')
        img = self.transform(img)

        # One-hot encode emotion
        emotion_onehot = torch.zeros(5)
        emotion_onehot[emotion_idx] = 1

        return img, emotion_onehot


# ==========================================
# Generator (256×256)
# ==========================================

class ConditionalGenerator(nn.Module):
    """
    Conditional Generator for 256×256 images
    """

    def __init__(self, z_dim=100, c_dim=5, ngf=64):
        super().__init__()

        self.z_dim = z_dim
        self.c_dim = c_dim

        # Initial size: 4×4
        # 4 → 8 → 16 → 32 → 64 → 128 → 256

        self.fc = nn.Sequential(
            nn.Linear(z_dim + c_dim, ngf * 32 * 4 * 4),
            nn.BatchNorm1d(ngf * 32 * 4 * 4),
            nn.ReLU(True)
        )

        self.main = nn.Sequential(
            # 4×4 → 8×8
            nn.ConvTranspose2d(ngf * 32, ngf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 16),
            nn.ReLU(True),

            # 8×8 → 16×16
            nn.ConvTranspose2d(ngf * 16, ngf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),

            # 16×16 → 32×32
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            # 32×32 → 64×64
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            # 64×64 → 128×128
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            # 128×128 → 256×256
            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z, c):
        """
        z: [batch, z_dim]
        c: [batch, c_dim] (one-hot)
        """
        # Concatenate z and c
        x = torch.cat([z, c], dim=1)

        # FC layer
        x = self.fc(x)
        x = x.view(x.size(0), -1, 4, 4)

        # Conv layers
        x = self.main(x)

        return x


# ==========================================
# Discriminator
# ==========================================

class ConditionalDiscriminator(nn.Module):
    """
    Conditional Discriminator for 256×256 images
    """

    def __init__(self, c_dim=5, ndf=64):
        super().__init__()

        # Image embedding
        self.img_conv = nn.Sequential(
            # 256×256 → 128×128
            nn.Conv2d(3, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 128×128 → 64×64
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # 64×64 → 32×32
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # 32×32 → 16×16
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # 16×16 → 8×8
            nn.Conv2d(ndf * 8, ndf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 16),
            nn.LeakyReLU(0.2, inplace=True),

            # 8×8 → 4×4
            nn.Conv2d(ndf * 16, ndf * 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 32),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # Emotion embedding
        self.emotion_embed = nn.Sequential(
            nn.Linear(c_dim, ndf * 32 * 4 * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Conv2d(ndf * 64, ndf * 32, 4, 1, 0, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 32, 1, 1, 1, 0, bias=False),
        )

    def forward(self, img, c):
        """
        img: [batch, 3, 256, 256]
        c: [batch, c_dim] (one-hot)
        """
        # Image features
        img_feat = self.img_conv(img)  # [batch, ndf*32, 4, 4]

        # Emotion features
        c_feat = self.emotion_embed(c)  # [batch, ndf*32*4*4]
        c_feat = c_feat.view(c_feat.size(0), -1, 4, 4)  # [batch, ndf*32, 4, 4]

        # Concatenate
        x = torch.cat([img_feat, c_feat], dim=1)  # [batch, ndf*64, 4, 4]

        # Classify
        x = self.classifier(x)  # [batch, 1, 1, 1]
        x = x.view(-1, 1)

        return x


# ==========================================
# Initialize
# ==========================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")

# Models
generator = ConditionalGenerator(z_dim=100, c_dim=5, ngf=64).to(device)
discriminator = ConditionalDiscriminator(c_dim=5, ndf=64).to(device)

print(f"✅ Generator parameters: {sum(p.numel() for p in generator.parameters()):,}")
print(f"✅ Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}")

# Optimizers
lr = 0.0002
beta1 = 0.5

g_optimizer = torch.optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))

# Loss
criterion = nn.BCEWithLogitsLoss()

print("✅ Models initialized!")

In [ ]:
# dataset = ConditionalEmotionDataset('/content/stylegan_data', img_size=256)

# dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2, drop_last=True)

# print(f"📊 Dataset size: {len(dataset)}")

# # Training config
# num_epochs = 100  # Start with 100, increase if needed
# z_dim = 100
# save_interval = 10

# # Fixed noise for visualization
# fixed_z = torch.randn(25, z_dim).to(device)
# fixed_c = torch.zeros(25, 5).to(device)
# for i in range(5):  # 5 emotions
#     for j in range(5):  # 5 samples each
#         fixed_c[i*5 + j, i] = 1

# # Training loop
# print("\n🏋️ Starting training...")

# for epoch in range(num_epochs):
#     epoch_g_loss = 0
#     epoch_d_loss = 0

#     pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")

#     for i, (real_imgs, emotions) in enumerate(pbar):
#         batch_size = real_imgs.size(0)

#         real_imgs = real_imgs.to(device)
#         emotions = emotions.to(device)

#         # Labels
#         real_labels = torch.ones(batch_size, 1).to(device)
#         fake_labels = torch.zeros(batch_size, 1).to(device)

#         # =====================================
#         # Train Discriminator
#         # =====================================

#         discriminator.zero_grad()

#         # Real images
#         real_validity = discriminator(real_imgs, emotions)
#         d_loss_real = criterion(real_validity, real_labels)

#         # Fake images
#         z = torch.randn(batch_size, z_dim).to(device)
#         fake_imgs = generator(z, emotions)
#         fake_validity = discriminator(fake_imgs.detach(), emotions)
#         d_loss_fake = criterion(fake_validity, fake_labels)

#         # Total D loss
#         d_loss = (d_loss_real + d_loss_fake) / 2
#         d_loss.backward()
#         d_optimizer.step()

#         # =====================================
#         # Train Generator
#         # =====================================

#         generator.zero_grad()

#         # Generate fake images
#         z = torch.randn(batch_size, z_dim).to(device)
#         fake_imgs = generator(z, emotions)
#         fake_validity = discriminator(fake_imgs, emotions)

#         # G loss (fool discriminator)
#         g_loss = criterion(fake_validity, real_labels)
#         g_loss.backward()
#         g_optimizer.step()

#         # Statistics
#         epoch_g_loss += g_loss.item()
#         epoch_d_loss += d_loss.item()

#         pbar.set_postfix({
#             'D_loss': f'{d_loss.item():.4f}',
#             'G_loss': f'{g_loss.item():.4f}'
#         })

#     # Epoch summary
#     avg_g_loss = epoch_g_loss / len(dataloader)
#     avg_d_loss = epoch_d_loss / len(dataloader)

#     print(f"\nEpoch {epoch+1}: D_loss={avg_d_loss:.4f}, G_loss={avg_g_loss:.4f}")

#     # Save samples
#     if (epoch + 1) % save_interval == 0:
#         generator.eval()
#         with torch.no_grad():
#             sample_imgs = generator(fixed_z, fixed_c)
#             sample_imgs = (sample_imgs + 1) / 2  # [-1,1] → [0,1]

#         # Display
#         import matplotlib.pyplot as plt

#         fig, axes = plt.subplots(5, 5, figsize=(12, 12))
#         emotions_list = ['Happy', 'Sad', 'Angry', 'Surprise', 'Neutral']

#         for i in range(5):
#             for j in range(5):
#                 idx = i * 5 + j
#                 img = sample_imgs[idx].cpu().permute(1, 2, 0).numpy()
#                 axes[i, j].imshow(img)
#                 axes[i, j].axis('off')
#                 if j == 0:
#                     axes[i, j].set_title(emotions_list[i], fontsize=14)

#         plt.suptitle(f'Generated Images - Epoch {epoch+1}', fontsize=16)
#         plt.tight_layout()
#         plt.show()

#         generator.train()

#     # Save checkpoint
#     if (epoch + 1) % 20 == 0:
#         torch.save({
#             'epoch': epoch,
#             'generator': generator.state_dict(),
#             'discriminator': discriminator.state_dict(),
#             'g_optimizer': g_optimizer.state_dict(),
#             'd_optimizer': d_optimizer.state_dict(),
#         }, f'/content/drive/MyDrive/cgan_checkpoint_epoch_{epoch+1}.pth')
#         print(f"💾 Checkpoint saved!")

# print("\n✅ Training complete!")

In [ ]:
"""
Complete Conditional GAN with R1 Penalty - Working Version
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import json
from tqdm import tqdm
import matplotlib.pyplot as plt

# ==========================================
# Dataset
# ==========================================

class ConditionalEmotionDataset(Dataset):
    """
    Dataset for conditional GAN
    """

    def __init__(self, data_path, img_size=256):
        self.data_path = data_path
        self.img_size = img_size

        # Load labels
        labels_file = os.path.join(data_path, 'dataset.json')
        with open(labels_file, 'r') as f:
            data = json.load(f)

        self.labels = data['labels']  # [[filename, emotion_idx], ...]

        self.transform = transforms.Compose([
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # [-1, 1]
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_name, emotion_idx = self.labels[idx]

        # Load image
        img_path = os.path.join(self.data_path, '00000', img_name)
        img = Image.open(img_path).convert('RGB')
        img = self.transform(img)

        # One-hot encode emotion
        emotion_onehot = torch.zeros(5)
        emotion_onehot[emotion_idx] = 1

        return img, emotion_onehot


# ==========================================
# Generator (256×256)
# ==========================================

class ConditionalGenerator(nn.Module):
    """
    Conditional Generator for 256×256 images
    """

    def __init__(self, z_dim=100, c_dim=5, ngf=64):
        super().__init__()

        self.z_dim = z_dim
        self.c_dim = c_dim

        self.fc = nn.Sequential(
            nn.Linear(z_dim + c_dim, ngf * 32 * 4 * 4),
            nn.BatchNorm1d(ngf * 32 * 4 * 4),
            nn.ReLU(True)
        )

        self.main = nn.Sequential(
            # 4×4 → 8×8
            nn.ConvTranspose2d(ngf * 32, ngf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 16),
            nn.ReLU(True),

            # 8×8 → 16×16
            nn.ConvTranspose2d(ngf * 16, ngf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),

            # 16×16 → 32×32
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            # 32×32 → 64×64
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            # 64×64 → 128×128
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            # 128×128 → 256×256
            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z, c):
        x = torch.cat([z, c], dim=1)
        x = self.fc(x)
        x = x.view(x.size(0), -1, 4, 4)
        x = self.main(x)
        return x


# ==========================================
# Discriminator
# ==========================================

class ConditionalDiscriminator(nn.Module):
    """
    Conditional Discriminator for 256×256 images
    """

    def __init__(self, c_dim=5, ndf=64):
        super().__init__()

        self.img_conv = nn.Sequential(
            # 256×256 → 128×128
            nn.Conv2d(3, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 128×128 → 64×64
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # 64×64 → 32×32
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # 32×32 → 16×16
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # 16×16 → 8×8
            nn.Conv2d(ndf * 8, ndf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 16),
            nn.LeakyReLU(0.2, inplace=True),

            # 8×8 → 4×4
            nn.Conv2d(ndf * 16, ndf * 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 32),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.emotion_embed = nn.Sequential(
            nn.Linear(c_dim, ndf * 32 * 4 * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )

        self.classifier = nn.Sequential(
            nn.Conv2d(ndf * 64, ndf * 32, 4, 1, 0, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 32, 1, 1, 1, 0, bias=False),
        )

    def forward(self, img, c):
        img_feat = self.img_conv(img)
        c_feat = self.emotion_embed(c)
        c_feat = c_feat.view(c_feat.size(0), -1, 4, 4)
        x = torch.cat([img_feat, c_feat], dim=1)
        x = self.classifier(x)
        x = x.view(-1, 1)
        return x


# ==========================================
# Training Function with R1
# ==========================================

def train_with_r1_penalty(
    discriminator,
    generator,
    real_imgs,
    emotions,
    d_optimizer,
    device,
    iteration,
    r1_gamma=10.0
):
    """
    Train discriminator with R1 regularization
    """

    batch_size = real_imgs.size(0)

    # Standard GAN loss
    real_validity = discriminator(real_imgs, emotions)
    d_loss_real = F.binary_cross_entropy_with_logits(
        real_validity,
        torch.ones_like(real_validity)
    )

    z = torch.randn(batch_size, 100).to(device)
    with torch.no_grad():
        fake_imgs = generator(z, emotions)
    fake_validity = discriminator(fake_imgs, emotions)
    d_loss_fake = F.binary_cross_entropy_with_logits(
        fake_validity,
        torch.zeros_like(fake_validity)
    )

    d_loss = d_loss_real + d_loss_fake

    # R1 Regularization (every 16 iterations)
    if iteration % 16 == 0:
        real_imgs.requires_grad = True
        real_validity_r1 = discriminator(real_imgs, emotions)

        # Compute gradients
        gradients = torch.autograd.grad(
            outputs=real_validity_r1.sum(),
            inputs=real_imgs,
            create_graph=True,
            retain_graph=True
        )[0]

        # R1 penalty
        r1_penalty = gradients.pow(2).reshape(batch_size, -1).sum(1).mean()

        # Add to loss
        d_loss = d_loss + r1_gamma * r1_penalty

        real_imgs.requires_grad = False

    d_optimizer.zero_grad()
    d_loss.backward()
    d_optimizer.step()

    return d_loss.item()


# ==========================================
# Main Training Script
# ==========================================

def main():
    # Config
    BATCH_SIZE = 8
    NUM_EPOCHS = 200
    LEARNING_RATE = 0.0002
    BETA1 = 0.5
    R1_GAMMA = 10.0
    SAVE_INTERVAL = 10


    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🖥️  Device: {device}")

    # Dataset
    dataset = ConditionalEmotionDataset('/content/stylegan_data', img_size=256)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2, drop_last=True)

    print(f"📊 Dataset size: {len(dataset)}")

    # Models
    generator = ConditionalGenerator(z_dim=100, c_dim=5, ngf=64).to(device)
    discriminator = ConditionalDiscriminator(c_dim=5, ndf=64).to(device)

    print(f"✅ Generator parameters: {sum(p.numel() for p in generator.parameters()):,}")
    print(f"✅ Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}")

    # Optimizers
    g_optimizer = torch.optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
    d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))

    # Loss
    criterion = nn.BCEWithLogitsLoss()

    # Fixed noise for visualization
    fixed_z = torch.randn(25, 100).to(device)
    fixed_c = torch.zeros(25, 5).to(device)
    for i in range(5):
        for j in range(5):
            fixed_c[i*5 + j, i] = 1

    # Training loop
    print("\n🏋️ Starting training with R1 penalty...")

    iteration = 0

    for epoch in range(NUM_EPOCHS):
        epoch_g_loss = 0
        epoch_d_loss = 0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

        for i, (real_imgs, emotions) in enumerate(pbar):
            batch_size = real_imgs.size(0)

            real_imgs = real_imgs.to(device)
            emotions = emotions.to(device)

            # =====================================
            # Train Discriminator with R1
            # =====================================

            d_loss = train_with_r1_penalty(
                discriminator, generator,
                real_imgs, emotions,
                d_optimizer, device,
                iteration, r1_gamma=R1_GAMMA
            )

            # =====================================
            # Train Generator
            # =====================================

            generator.zero_grad()

            z = torch.randn(batch_size, 100).to(device)
            fake_imgs = generator(z, emotions)
            fake_validity = discriminator(fake_imgs, emotions)

            g_loss = F.binary_cross_entropy_with_logits(
                fake_validity,
                torch.ones_like(fake_validity)
            )

            g_loss.backward()
            g_optimizer.step()

            # Statistics
            epoch_g_loss += g_loss.item()
            epoch_d_loss += d_loss

            pbar.set_postfix({
                'D_loss': f'{d_loss:.4f}',
                'G_loss': f'{g_loss.item():.4f}',
                'Iter': iteration
            })

            iteration += 1

        # Epoch summary
        avg_g_loss = epoch_g_loss / len(dataloader)
        avg_d_loss = epoch_d_loss / len(dataloader)

        print(f"\nEpoch {epoch+1}: D_loss={avg_d_loss:.4f}, G_loss={avg_g_loss:.4f}")

        # Save samples
        if (epoch + 1) % SAVE_INTERVAL == 0:
            generator.eval()
            with torch.no_grad():
                sample_imgs = generator(fixed_z, fixed_c)
                sample_imgs = (sample_imgs + 1) / 2

            # Display
            fig, axes = plt.subplots(5, 5, figsize=(12, 12))
            emotions_list = ['Happy', 'Sad', 'Angry', 'Surprise', 'Neutral']

            for i in range(5):
                for j in range(5):
                    idx = i * 5 + j
                    img = sample_imgs[idx].cpu().permute(1, 2, 0).numpy()
                    axes[i, j].imshow(img)
                    axes[i, j].axis('off')
                    if j == 0:
                        axes[i, j].set_title(emotions_list[i], fontsize=14)

            plt.suptitle(f'Generated Images with R1 - Epoch {epoch+1}', fontsize=16)
            plt.tight_layout()
            plt.show()

            generator.train()

        # Save checkpoint
        if (epoch + 1) % 20 == 0:
            torch.save({
                'epoch': epoch,
                'generator': generator.state_dict(),
                'discriminator': discriminator.state_dict(),
                'g_optimizer': g_optimizer.state_dict(),
                'd_optimizer': d_optimizer.state_dict(),
            }, f'/content/drive/MyDrive/cgan_r1_checkpoint_epoch_{epoch+1}.pth')
            print(f"💾 Checkpoint saved!")

    print("\n✅ Training complete!")


if __name__ == '__main__':
    main()

In [16]:
# # Create conditional_stylegan2.py
# import torch, torchvision
# print(torch.__version__, torchvision.__version__)

# import torch.nn as nn
# from torch_utils import misc
# from torch_utils.ops import conv2d_resample
# from torch_utils.ops import upfirdn2d

# class ConditionalGenerator(nn.Module):
#     """
#     StyleGAN2 Generator with emotion conditioning
#     """

#     def __init__(
#         self,
#         z_dim=512,           # Latent vector size
#         c_dim=5,             # Emotion classes
#         w_dim=512,           # Intermediate latent
#         img_resolution=256,  # Output resolution
#         img_channels=3,      # RGB
#         **kwargs
#     ):
#         super().__init__()

#         # Mapping Network (with conditioning)
#         self.mapping = MappingNetwork(
#             z_dim=z_dim,
#             c_dim=c_dim,
#             w_dim=w_dim,
#             num_layers=8
#         )

#         # Synthesis Network
#         self.synthesis = SynthesisNetwork(
#             w_dim=w_dim,
#             img_resolution=img_resolution,
#             img_channels=img_channels,
#             **kwargs
#         )

#     def forward(self, z, c, truncation_psi=1):
#         """
#         z: latent code [batch, z_dim]
#         c: emotion label (one-hot) [batch, c_dim]
#         """
#         # Map z + c → w
#         w = self.mapping(z, c, truncation_psi=truncation_psi)

#         # Generate image
#         img = self.synthesis(w)

#         return img


# class MappingNetwork(nn.Module):
#     """
#     Maps z + emotion → w (style vectors)
#     """

#     def __init__(self, z_dim, c_dim, w_dim, num_layers=8):
#         super().__init__()

#         # Embed emotion label
#         self.embed = nn.Linear(c_dim, z_dim)

#         # Mapping layers
#         layers = []
#         for i in range(num_layers):
#             in_dim = z_dim if i == 0 else w_dim
#             layers.append(nn.Linear(in_dim, w_dim))
#             layers.append(nn.LeakyReLU(0.2))

#         self.net = nn.Sequential(*layers)

#     def forward(self, z, c, truncation_psi=1):
#         # Embed emotion
#         c_embed = self.embed(c)

#         # Concatenate z + c_embed
#         x = z + c_embed  # Element-wise addition

#         # Map to w
#         w = self.net(x)

#         # Truncation trick
#         if truncation_psi < 1:
#             w_avg = self.w_avg  # Learned average w
#             w = w_avg + truncation_psi * (w - w_avg)

#         return w

In [17]:
# # Create conditional_stylegan2.py

# import torch
# import torch.nn as nn
# from torch_utils import misc
# from torch_utils.ops import conv2d_resample
# from torch_utils.ops import upfirdn2d

# class ConditionalGenerator(nn.Module):
#     """
#     StyleGAN2 Generator with emotion conditioning
#     """

#     def __init__(
#         self,
#         z_dim=512,           # Latent vector size
#         c_dim=5,             # Emotion classes
#         w_dim=512,           # Intermediate latent
#         img_resolution=256,  # Output resolution
#         img_channels=3,      # RGB
#         **kwargs
#     ):
#         super().__init__()

#         # Mapping Network (with conditioning)
#         self.mapping = MappingNetwork(
#             z_dim=z_dim,
#             c_dim=c_dim,
#             w_dim=w_dim,
#             num_layers=8
#         )

#         # Synthesis Network
#         self.synthesis = SynthesisNetwork(
#             w_dim=w_dim,
#             img_resolution=img_resolution,
#             img_channels=img_channels,
#             **kwargs
#         )

#     def forward(self, z, c, truncation_psi=1):
#         """
#         z: latent code [batch, z_dim]
#         c: emotion label (one-hot) [batch, c_dim]
#         """
#         # Map z + c → w
#         w = self.mapping(z, c, truncation_psi=truncation_psi)

#         # Generate image
#         img = self.synthesis(w)

#         return img


# class MappingNetwork(nn.Module):
#     """
#     Maps z + emotion → w (style vectors)
#     """

#     def __init__(self, z_dim, c_dim, w_dim, num_layers=8):
#         super().__init__()

#         # Embed emotion label
#         self.embed = nn.Linear(c_dim, z_dim)

#         # Mapping layers
#         layers = []
#         for i in range(num_layers):
#             in_dim = z_dim if i == 0 else w_dim
#             layers.append(nn.Linear(in_dim, w_dim))
#             layers.append(nn.LeakyReLU(0.2))

#         self.net = nn.Sequential(*layers)

#     def forward(self, z, c, truncation_psi=1):
#         # Embed emotion
#         c_embed = self.embed(c)

#         # Concatenate z + c_embed
#         x = z + c_embed  # Element-wise addition

#         # Map to w
#         w = self.net(x)

#         # Truncation trick
#         if truncation_psi < 1:
#             w_avg = self.w_avg  # Learned average w
#             w = w_avg + truncation_psi * (w - w_avg)

#         return w

In [18]:
# # train_conditional_stylegan2.py

# import torch
# import torch.nn.functional as F
# from torch.utils.data import DataLoader
# from torchvision import transforms
# from tqdm import tqdm

# # Config
# BATCH_SIZE = 16  # Adjust based on GPU
# IMG_SIZE = 256
# EPOCHS = 50000
# LR = 0.002
# BETA1 = 0.0
# BETA2 = 0.99

# # Setup
# device = torch.device('cuda')

# # Load pretrained FFHQ
# generator = load_pretrained_ffhq('ffhq.pkl')

# # Convert to conditional
# generator = make_conditional(generator, num_classes=5)
# generator = generator.to(device)

# # Discriminator
# discriminator = ConditionalDiscriminator(
#     img_resolution=256,
#     c_dim=5
# ).to(device)

# # Optimizers
# g_optimizer = torch.optim.Adam(
#     generator.parameters(),
#     lr=LR,
#     betas=(BETA1, BETA2)
# )

# d_optimizer = torch.optim.Adam(
#     discriminator.parameters(),
#     lr=LR,
#     betas=(BETA1, BETA2)
# )

# # Dataset
# dataset = HijabiEmotionDataset('/content/drive/MyDrive/data')
# dataloader = DataLoader(
#     dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     num_workers=2
# )

# # Training Loop
# for epoch in range(EPOCHS):

#     for real_imgs, emotions in tqdm(dataloader):
#         batch_size = real_imgs.size(0)

#         real_imgs = real_imgs.to(device)
#         emotions = emotions.to(device)  # One-hot

#         # =====================================
#         # Train Discriminator
#         # =====================================

#         # Real images
#         real_validity = discriminator(real_imgs, emotions)
#         d_loss_real = F.softplus(-real_validity).mean()

#         # Fake images
#         z = torch.randn(batch_size, 512).to(device)
#         fake_imgs = generator(z, emotions)
#         fake_validity = discriminator(fake_imgs.detach(), emotions)
#         d_loss_fake = F.softplus(fake_validity).mean()

#         # R1 regularization (every 16 iterations)
#         if epoch % 16 == 0:
#             real_imgs.requires_grad = True
#             real_validity = discriminator(real_imgs, emotions)
#             r1_penalty = compute_r1_penalty(real_validity, real_imgs)
#             d_loss = d_loss_real + d_loss_fake + r1_penalty * 10
#         else:
#             d_loss = d_loss_real + d_loss_fake

#         d_optimizer.zero_grad()
#         d_loss.backward()
#         d_optimizer.step()

#         # =====================================
#         # Train Generator
#         # =====================================

#         z = torch.randn(batch_size, 512).to(device)
#         fake_imgs = generator(z, emotions)
#         fake_validity = discriminator(fake_imgs, emotions)

#         g_loss = F.softplus(-fake_validity).mean()

#         g_optimizer.zero_grad()
#         g_loss.backward()
#         g_optimizer.step()

#     # Logging
#     if epoch % 100 == 0:
#         print(f"Epoch {epoch} | D: {d_loss.item():.4f} | G: {g_loss.item():.4f}")

#         # Save samples
#         save_samples(generator, epoch)

#     # Save checkpoint
#     if epoch % 1000 == 0:
#         save_checkpoint(generator, discriminator, epoch)